# Homework: Agentic RAG

In this homework, we build a RAG system from scratch and then make it
agentic - the same path as the module.

Instead of the course FAQ, our knowledge base is the course lessons
themselves.


## Preparation

In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from openai import OpenAI
openai_client = OpenAI()

In [3]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [4]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

## Question #1. How many lesson pages

How many lesson pages are in the dataset?

* 24
* 72
* 240
* 720

In [5]:
print(f"The number of files is: {len(files)}.")

The number of files is: 72.


> **Answer**: The number of files is: 72.


## Question #2. Indexing and searching

Index the documents with minsearch - make `content` a text field and
`filename` a keyword field. Then search with this query:

> How does the agentic loop keep calling the model until it stops?

What's the `filename` of the first result?

* `01-agentic-rag/lessons/03-rag.md`
* `01-agentic-rag/lessons/14-agentic-loop.md`
* `04-evaluation/lessons/13-llm-as-judge.md`
* `06-best-practices/lessons/02-hybrid-search.md`

In [6]:
import minsearch

index = minsearch.Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)
index.fit(documents)

In [7]:
query = "How does the agentic loop keep calling the model until it stops?"

search_results = index.search(
    query=query,
    filter_dict={},
    boost_dict={"content": 1.0},
    num_results=5,
)

In [8]:
print(f"First result is: {search_results[0]["filename"]}")

First result is: 01-agentic-rag/lessons/14-agentic-loop.md


> **Answer**: `01-agentic-rag/lessons/14-agentic-loop.md`

## Question #3. RAG

Now we will build a RAG assistant on top of this data. Let's use the rag helper 
script we prepared during the lessons:

```bash
wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py
```

`RAGBase` was written for the FAQ schema (`section`/`question`/`answer`),
while our documents have `filename` and `content`.

Two solutions are possible:

- Implement the RAG flow yourself
- Take `RAGBase` and change the parts related to the FAQ schema - `search` (to use our index) and `build_context`

Build a RAG over the index from Q2 and answer the query:

> How does the agentic loop keep calling the model until it stops?

Use gpt-5.4-mini. How many input (prompt) tokens did we send to the model for
this request?

* 700
* 7000
* 70000
* 700000

We count input tokens instead of price because the cost depends on the model
and provider you use, but the size of the prompt we send is the same for
everyone.

Most LLM APIs report token usage on the response object (e.g.
`response.usage.input_tokens` / `prompt_tokens`). We'll read the input tokens
from there.

You will need to modify the code for the rag helper to expose the usage.

In the RAG Helper class, `llm` returns only the text. Modify it to return the whole response, and change `rag` to return both the answer and usage (as a tuple or create a small dataclass for that).

Note: for this question and the next ones, if your answer doesn't match exactly,
just select the closest option - especially if you use a different model or a
different LLM provider.


In [28]:
import importlib
import rag_helper
importlib.reload(rag_helper)
from rag_helper import RAGBase

rag = RAGBase(index, openai_client)
query = "How does the agentic loop keep calling the model until it stops?"
results = rag.rag(query)


Input tokens: 7036
Output tokens: 131


> **Answer:** Input tokens were 7036, approx 7000 tokens.